In [1]:
%env SPECTRAL_CONNECTIVITY_ENABLE_GPU=true


env: SPECTRAL_CONNECTIVITY_ENABLE_GPU=true


In [2]:
import os
os.chdir('/blue/npadillacoreano/mcum/SocialMemEphys/diff_fam_social_memory_ephys')
os.environ['CUDA_PATH'] = '/apps/compilers/cuda/13.0.2'
import spikeinterface.extractors as se 
import cupy as xp
from cupyx.scipy.fft import ifft
from cupyx.scipy.sparse.linalg import svds
from spectral_connectivity import Multitaper, Connectivity
import importlib
import pandas as pd
from itertools import combinations
import os
from bidict import bidict
import lfp.lfp_analysis.LFP_collection as LFP_collection
import lfp.lfp_analysis.LFP_recording as LFP_recording
import pickle 
import numpy as np
def pickle_this(thing_to_pickle, file_name):
    """
    Pickles things
    Args (2):
        thing_to_pickle: anything you want to pickle
        file_name: str, filename that ends with .pkl
    Returns:
        none
    """
    with open(file_name, "wb") as file:
        pickle.dump(thing_to_pickle, file)
def unpickle_this(pickle_file):
    """
    Unpickles things
    Args (1):
        file_name: str, pickle filename that already exists and ends with .pkl
    Returns:
        pickled item
    """
    with open(pickle_file, "rb") as file:
        return pickle.load(file)

df = pd.read_excel(r"lfp/channel_mapping_sme.xlsx")
spike_cols = [col for col in df.columns if 'spike_interface_' in col.lower()]

# Extract brain regions from column names
# Assumes format 'spike_interface_REGION'
brain_regions = [col.split('spike_interface_')[1] for col in spike_cols]

# Create nested dictionary
subject_to_channel_dict = {}

data_path = r"data/rehouse22_23_data"




target_dict = {'1.1': ['BLA'],
              '1.2': [''],
              '1.3': ['BLA'],
              '2.1': [],
              '2.2': ['vHPC'],
              '2.3': ['mPFC', 'vHPC'],
              '2.4': ['BLA','MD'],
              '3.1': [],
              '3.2': [],
              '3.3': ['NAc', 'MD', 'vHPC'],
              '4.1': [],
              '4.4': ['NAc']}



for _, row in df.iterrows():
    subject = row['Subject'].astype(str)
    # Initialize inner dictionary for this subject
    subject_to_channel_dict[subject] = {}
    
    # Populate inner dictionary with brain region: spike value pairs
    for col, region in zip(spike_cols, brain_regions):
        subject_to_channel_dict[subject][region] = int(row[col])

behavior_dicts = {}
def make_recording_to_subj_dict(data_path):
    recording_to_subject = {}
    for root, dirs, files in os.walk(data_path):
        for file in files:
            if file.endswith('merged.rec'):
                subject = str(int((file.split("_")[0]))/10)
                recording_to_subject[file] = subject
                behavior_dicts[file] = {}
    return recording_to_subject

def process(data_path):
    recording_to_subject = make_recording_to_subj_dict(data_path)
    print(recording_to_subject)
    collection = LFP_collection.LFPCollection(subject_to_channel_dict, data_path, recording_to_subject, 5)
    collection.preprocess()
    return collection    
    


In [3]:
collection = process(data_path)

{'23_rehouse_d4_merged.rec': '2.3', '22_rehouse_d4_merged.rec': '2.2', '23_rehouse_d1_merged.rec': '2.3', '22_rehouse_d1_merged.rec': '2.2', '23_rehouse_d2_merged.rec': '2.3', '22_rehouse_d5_merged.rec': '2.2', '23_rehouse_d5_merged.rec': '2.3', '23_rehouse_d0_merged.rec': '2.3', '22_rehouse_d0_merged.rec': '2.2', '22_rehouse_d2_merged.rec': '2.2', '22_rehouse_d7_merged.rec': '2.2', '23_rehouse_d7_merged.rec': '2.3', '23_rehouse_d3_merged.rec': '2.3', '22_rehouse_d3_merged.rec': '2.2', '23_rehouse_d6_merged.rec': '2.3', '22_rehouse_d6_merged.rec': '2.2'}
Processing 23_rehouse_d4_merged.rec
Processing 22_rehouse_d4_merged.rec
Processing 23_rehouse_d1_merged.rec
Processing 22_rehouse_d1_merged.rec
Processing 23_rehouse_d2_merged.rec
Processing 22_rehouse_d5_merged.rec
Processing 23_rehouse_d5_merged.rec
Processing 23_rehouse_d0_merged.rec
Processing 22_rehouse_d0_merged.rec
Processing 22_rehouse_d2_merged.rec
Processing 22_rehouse_d7_merged.rec
Processing 23_rehouse_d7_merged.rec
Process

  0%|          | 0/16 [00:00<?, ?it/s]

processing 23_rehouse_d4_merged.rec


  6%|▋         | 1/16 [00:00<00:08,  1.72it/s]

RMS Traces calculated
processing 22_rehouse_d4_merged.rec


...

RMS Traces calculated
processing 23_rehouse_d6_merged.rec


 94%|█████████▍| 15/16 [00:07<00:00,  2.20it/s]

RMS Traces calculated
processing 22_rehouse_d6_merged.rec


100%|██████████| 16/16 [00:07<00:00,  2.13it/s]

RMS Traces calculated


In [4]:
collection.calculate_all()

  0%|          | 0/16 [00:01<?, ?it/s]


CUDADriverError: CUDA_ERROR_NO_BINARY_FOR_GPU: no kernel image is available for execution on the device

In [4]:
collection.exclude_regions(target_dict)                        
save_to_json(collection, "/data/rehouse_lfp_22_23")